In [ ]:

from py4stat import datania
import pandas as pd
import math
from collections import defaultdict

# Task 1: Initialize running totals
# We need to track these for EACH province separately
province_stats = defaultdict(lambda: {
    'income_sum': 0,
    'income_sum_sq': 0,
    'count': 0
})

# Task 2: Process in chunks
csv_file = datania.generate_labour_force_survey(n_persons=5000, seed=42)

for chunk in pd.read_csv(csv_file, chunksize=500):
    # Filter to employed persons with valid income
    employed = chunk[
        (chunk['employment_status'] == 'Employed') &
        (chunk['monthly_income'].notna())
    ]

    # Update running totals for each province in this chunk
    for province in employed['province'].unique():
        province_data = employed[employed['province'] == province]
        incomes = province_data['monthly_income']

        # YOUR CODE HERE: Update the running totals
        # province_stats[province]['income_sum'] += ???
        # province_stats[province]['income_sum_sq'] += ???
        # province_stats[province]['count'] += ???

print(f"Processed data for {len(province_stats)} provinces")


# Task 3: Calculate final statistics
results = []

for province, stats in province_stats.items():
    if stats['count'] > 0:
        mean_income = stats['income_sum'] / stats['count']

        # Standard deviation formula: sqrt(E[X²] - E[X]²)
        variance = (stats['income_sum_sq'] / stats['count']) - (mean_income ** 2)
        std_income = math.sqrt(max(0, variance))  # max(0, ...) handles floating point errors

        results.append({
            'province': province,
            'employed_count': stats['count'],
            'mean_income': round(mean_income, 2),
            'std_income': round(std_income, 2)
        })

df_results = pd.DataFrame(results).sort_values('province')
print("\n=== CHUNKED RESULTS ===")
print(df_results.to_string(index=False))


# Task 4: Verify against direct calculation
csv_file = datania.generate_labour_force_survey(n_persons=5000, seed=42)
df_full = pd.read_csv(csv_file)
employed_full = df_full[
    (df_full['employment_status'] == 'Employed') &
    (df_full['monthly_income'].notna())
]

print("\n=== VERIFICATION (Direct Pandas) ===")
verification = employed_full.groupby('province')['monthly_income'].agg(['count', 'mean', 'std'])
print(verification.round(2))

# YOUR CODE HERE: Compare the results - do they match?